# Squeal — примеры по этапам

Kernel: **Squeal (ee1)**. Ячейка — синтаксис Python, исполняется на BEAM:
Python её не запускает, `squeal` компилирует её в Abstract Erlang Forms и гоняет
в persistent Erlang-REPL (bindings живут между ячейками).

Некоторые ячейки ниже намеренно ошибочны — так показаны правила языка.

## Этап 1 — скелет: `def` и строка

In [4]:
def hello():
    "hi"

hello()

<<"hi">>


## Этап 2 — значения и выражения

Литералы: число, float, строка, атом (`'ok'`), bool, `None`:

In [5]:
(42, 1.5, "hi", 'ok', True, False, None)

{42,1.5,<<"hi">>,ok,true,false,nil}


Коллекции: список, кортеж, словарь (атом-ключ `'a'` и строка-ключ `"b"`):

In [6]:
([1, 2, 3], (1, 2), {'a': 1}, {"b": 2})

{[1,2,3],{1,2},#{a => 1},#{<<"b">> => 2}}


Операторы — Python-семантика для `// % **` и `==`:

In [7]:
(7 / 2, 7 // 2, -7 // 2, -7 % 2, 2 ** 10, 2 ** -1, 1 == 1.0, "a" < "b")

squeal runtime error: {error,undef}


Вызовы:

In [8]:
def add(a, b):
    a + b

add(2, 3)

5


## Этап 3 — присваивание и неизменяемость

Переприсваивание разрешено (новое Erlang-имя `X@1`):

In [9]:
x = 1
x = x + 1
x

2


Деструктуризация списка со звездой и кортежа:

In [10]:
head, *rest = [1, 2, 3]
(head, rest)

{1,[2,3]}


In [11]:
a, b = (10, 20)
a + b

30


Слияние словарей `|=` (порядок ключей в выводе map не гарантирован):

In [12]:
m = {'a': 1}
m |= {'b': 2}
(m == {'a': 1, 'b': 2}, m)

{true,#{a => 1,b => 2}}


Мутаций нет: `for` / `while` / `del` / `+=` — ошибка компиляции (ячейка ниже намеренно падает):

In [13]:
for i in [1, 2]:
    i

squeal: 1: цикл For запрещён — squeal без мутаций


## Этап 4 — ветвления

`if`/`else` — истинно только `true`:

In [16]:
if 1 < 2:
    r = 'yes'
else:
    r = 'no'
r

yes


`match`/`case` — or-образцы, guard, захват:

In [17]:
match 3:
    case 1 | 2:
        r = 'one or two'
    case n if n > 10:
        r = 'big'
    case _:
        r = 'other'
r

other


Образец списка:

In [22]:
match [1, 2, 3]:
    case [h, *t]:
        r = (h, t)
r

{1,[2,3]}


Неисчерпываемый `match` — «let it crash» (ячейка ниже падает в рантайме):

In [19]:
match 5:
    case 1:
        r = 'one'

squeal runtime error: {error,{case_clause,5}}


## Этап 5 — классы-типы

`class T(StartingWithAtom)` → конструктор инлайнится в кортеж `{t, …}`:

In [2]:
class CustomType(StartingWithAtom):
    x: int
    y: int

CustomType(1, 2)

{custom_type,1,2}


In [2]:
def fun(x):
    'ok', x

Ok(val) = fun(3)
val

3


aОбразец класса:

In [ ]:
match CustomType(1, 2):
    case CustomType(a, b):
        r = (a, b)
r

Встроенные `Ok`/`Error` — прелюдия, как в Gleam:

In [3]:
(Ok(1), Error("boom"))

{{ok,1},{error,<<"boom">>}}


In [4]:
match Ok(42):
    case Ok(v):
        r = v
    case Error(e):
        r = 'error'
r

42


## Этап 6 — лямбды и включения

`lambda` → `fun`, вызов связанного имени:

In [5]:
double_inc = lambda x: x + 1
double_inc(41)

42


Декораторы — как в Python, применяются один раз; цепочка — изнутри наружу:

In [ ]:
def plus1(fn):
    lambda x: fn(x) + 1

@plus1
@plus1
def z(x):
    x

z(1)

In [6]:
[x * 2 for x in [1, 2, 3]]

[2,4,6]


In [7]:
[x for x in [1, 2, 3, 4] if x > 2]

[3,4]


In [8]:
[x + y for x in [1, 2] for y in [10, 20]]

[11,21,12,22]


Dict comprehensions — ключи и пары через `.items()`:

In [1]:
{k: 1 for k in {'a': 9}}

erl: #{K => 1 || K <- squeal_iter(#{a => 9})}
#{a => 1}


In [3]:
{k: v * 2 for k, v in {'a': 1}.items()}

erl: #{K => V * 2 || {K, V} <- squeal_iter(maps:to_list(#{a => 1}))}
#{a => 2}


In [2]:
{'a': 1}.items()

[{a,1}]


## Этап 7 — модули и импорты

Импорты — часть модульной сборки (`src/`, `python -m squeal build`), в ноутбуке
внешних модулей нет. Компилятор строит удалённый вызов `a@b@c:helper(...)`,
и он упадёт с `undef`, потому что модуль не загружен (ячейка ниже падает):

In [4]:
from a.b.c import helper
helper()

squeal runtime error: {error,undef}


## Этап 8 — паритет с Gleam

18 пар эквивалентных модулей проверяют, что squeal и Gleam 1.19 порождают
одинаковые Abstract Erlang Forms. Это тесты репозитория (`pytest tests/test_parity.py`),
в ноутбуке не демонстрируется.

## Этап 9 — ошибки с позицией

Ошибка компиляции печатается как `squeal: <строка>: <текст>`
(в файловой сборке — `squeal: src/x.ee:2: …`). Две намеренно ошибочные ячейки:

In [9]:
if True:
    v = 1

squeal: 1: if без else — нет значения на ложном пути


In [10]:
[1, 2].append(3)

squeal: 1: метод 'append' не поддержан (только .items())


In [1]:
def fib(n, current, next):
    match n:
        case 0:
            current

        case _ if n > 0:
            fib(n - 1, next, current + next)


def fib(n):
    fib(n, 0, 1)


In [6]:
fib(7)

13


In [1]:
def dec(f):
    def fun(x):
        x


@dec
def g():
    return 1

g(3)

squeal: 10: функция g/1 не определена (есть g/0)


In [3]:
def dec(f):
    def fun(x):
        100


@dec
def g(x):
    x * 2


g(1)


100


In [9]:
"dggergeg".startswith("dg")

true


In [2]:
{k: k for _i, k in enumerate(range(4))}

erl: #{K => K || {_i, K} <- squeal_iter(squeal_enumerate(0, squeal_range(0, 4, 1)))}
#{0 => 0,1 => 1,2 => 2,3 => 3}
